# La brecha del ICFES en Caldas

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo. Ustedes lo siguen, lo corren y me interrumpen. Termina, como *El IPC del sancocho*, en un carrusel para LinkedIn.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Limpiar un dato de personas dejando cada decisión a la vista, escoger una métrica que responda a la decisión del cliente, revisar a quién le falla el modelo y contar el resultado. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular), 2 (seleccionar y preparar datos), 4 (implementar y evaluar), 5 (comunicar) y 6 (reflexión ética). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea o que escriba una gráfica. Deben poder explicar sin ayuda por qué la exactitud engaña aquí, qué es la captación y por qué la partición se hace por colegio. |
| **Entregable** | Este cuaderno corrido en su portafolio, con las predicciones y la decisión en tres frases. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path.cwd() / "taller-cd-ml-2026-2"] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

PALETA = ["#1D4E89", "#D7263D", "#1B998B", "#6C4AB6", "#5BC0EB", "#F49097", "#9AA5B1"]      # colores del curso
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETA), "axes.titleweight": "bold", "figure.dpi": 110})
print("Herramientas listas.")

## 1. La pregunta

La Secretaría de Educación de Caldas tiene un programa de refuerzo para grado once. El presupuesto alcanza para uno de cada cuatro estudiantes, y los cupos se reparten antes del examen, cuando solo se sabe lo que cada uno respondió en la inscripción.

1. **Predecir.** ¿Qué estudiantes van a quedar en el cuarto más bajo del país?
2. **Explicar.** ¿Qué condiciones del hogar y del colegio acompañan ese resultado?
3. **Decidir.** Con cupo para el 25 %, ¿a quiénes se les ofrece el refuerzo, y a quiénes deja por fuera esa regla?

**El dato.** Saber 11, aplicación 2020-2, publicado por el ICFES en datos.gov.co. Cada fila es un estudiante anonimizado, con su puntaje, sus respuestas sobre el hogar y los datos de su colegio.

**La respuesta.** $y_i=1$ si el percentil nacional del estudiante $i$ es 25 o menos.

**La métrica.** Si el modelo ordena a los estudiantes por riesgo y el programa atiende al primer 25 %, la **captación** es la fracción de los que quedaron abajo que recibió cupo:

$$\text{captación}=\frac{\#\{i:\ y_i=1\ \text{y}\ i\ \text{recibe cupo}\}}{\#\{i:\ y_i=1\}}.$$

### Predicción escrita

- En Caldas, ¿qué fracción de los estudiantes queda en el cuarto más bajo del país? **Mi apuesta:**
- ¿Qué pesa más: el colegio, la zona o la familia? **Mi apuesta:**
- Si los cupos se reparten por sorteo, ¿cuál es la captación? **Mi apuesta:**

In [ ]:
from taller.icfes import preparar, tasa_por, tramos, competir, asignar_cupos, auditar

DEPARTAMENTO = "CALDAS"      # al final pueden cambiarlo por otro departamento
CUPO = 0.25
crudo = descargar("saber11", departamento=DEPARTAMENTO)
print(f"{len(crudo):,} estudiantes de {crudo['cole_cod_dane_establecimiento'].nunique()} colegios".replace(",", "."))

## 2. Tres trampas antes de modelar

**Edades imposibles.** La fecha de nacimiento trae errores de digitación.

In [ ]:
edad = 2020 - pd.to_numeric(crudo["estu_fechanacimiento"].astype(str).str[:4], errors="coerce")
print("Edades fuera de 13 a 70 años:", int((~edad.between(13, 70)).sum()))

**Puntajes en cero.** Un puntaje global de cero es sospechoso en un examen de cinco pruebas. Miremos cuántos hay y en qué estado los publica el ICFES.

In [ ]:
puntaje = pd.to_numeric(crudo["punt_global"], errors="coerce")
print("Puntaje global igual a cero:", int((puntaje == 0).sum()))
print(crudo.loc[puntaje == 0, "estu_estadoinvestigacion"].value_counts().to_string())

**Faltantes que no son al azar.** Quienes dejan preguntas en blanco no se parecen a los demás. Si borráramos esas filas, la muestra quedaría más privilegiada de lo que es. Por eso no se borran: los modelos llenan los huecos con la mediana del entrenamiento, y lo decimos.

`preparar` convierte el formulario en números, deja las edades imposibles como faltantes y quita solo los puntajes en cero, porque no describen un desempeño. Es una decisión, y va a la bitácora.

In [ ]:
X, y, colegio, datos = preparar(crudo)
incompleta = X.isna().any(axis=1)
print(f"Quedan {len(X):,} estudiantes. En el cuarto más bajo del país: {y.mean() * 100:.1f} %".replace(",", "."))
print(f"Con algún faltante: {incompleta.mean() * 100:.1f} % | de ellos, en el cuarto más bajo: {y[incompleta].mean() * 100:.1f} %")

## 3. La brecha

Antes de cualquier modelo, la pregunta descriptiva: ¿en qué grupos se concentran los puntajes bajos? Las barras llevan dos errores estándar, para no leer diferencias donde hay ruido.

In [ ]:
madre = {"Ninguno": "ninguna", "Primaria incompleta": "primaria", "Primaria completa": "primaria",
         "Secundaria (Bachillerato) incompleta": "bachillerato", "Secundaria (Bachillerato) completa": "bachillerato",
         "Técnica o tecnológica incompleta": "técnica", "Técnica o tecnológica completa": "técnica",
         "Educación profesional incompleta": "profesional", "Educación profesional completa": "profesional",
         "Postgrado": "posgrado"}
paneles = {
    "Colegio y zona": tasa_por(y, datos["cole_naturaleza"].str.capitalize() + ", " + datos["cole_area_ubicacion"].str.lower()),
    "Educación de la madre": tasa_por(y, datos["fami_educacionmadre"].map(madre),
                                      ["ninguna", "primaria", "bachillerato", "técnica", "profesional", "posgrado"]),
    "Internet en la casa": tasa_por(y, datos["fami_tieneinternet"].map({"Si": "sí", "No": "no"})),
    "Jornada": tasa_por(y, datos["cole_jornada"].str.lower()),
}
fig, ejes = plt.subplots(1, 4, figsize=(17, 3.8))
for eje, (titulo, tabla) in zip(ejes, paneles.items()):
    eje.barh(tabla.index, tabla["tasa"], xerr=2 * tabla["ee"], color=PALETA[0], ecolor="#9AA5B1")
    eje.axvline(y.mean() * 100, color="#D7263D", ls=":"); eje.set(title=titulo, xlabel="% en el cuarto más bajo")
plt.tight_layout(); plt.show()

In [ ]:
municipios = tasa_por(y, datos["cole_mcpio_ubicacion"].str.title())
municipios = municipios[municipios["n"] >= 80]
print("Municipios con 80 estudiantes o más, del más alto al más bajo en el cuarto inferior del país:")
print(municipios.sort_values("tasa", ascending=False)[["n", "tasa"]].round(1).to_string())

**Lectura.** Las brechas más grandes aparecen en lo que hay en la casa y en dónde queda el colegio. La diferencia entre colegio oficial y privado es menor. La jornada sabatina y la nocturna agrupan a muchos adultos que validan el bachillerato, y por eso conviene leerlas aparte. Ninguna de estas barras es una causa: son grupos que se superponen.

## 4. Predecir: ¿a quién le llega el cupo?

**Los tramos se parten por colegio.** Dos estudiantes del mismo colegio se parecen. Si uno queda en entrenamiento y el otro en prueba, el modelo parece mejor de lo que es. Por eso cada colegio va entero a un solo tramo: 60 % para entrenar, 20 % para escoger y 20 % para reportar.

Compiten cinco candidatos: el sorteo, una regla de una sola columna (la educación de la madre) y tres modelos (regresión logística, un árbol y un bosque aleatorio).

In [ ]:
E, V, P = tramos(X, y, colegio)
todos, puntajes = competir(X, y, E, V, P, CUPO)
display(todos.sort_values("captación en validación", ascending=False).round(3))
print(f"Techo de la captación con cupo del {CUPO:.0%}: {min(1, CUPO / y[P].mean()):.2f}")

In [ ]:
mejor = todos.drop(index="sorteo")["captación en validación"].idxmax()
regla = "regla: educación de la madre"
elegido = mejor if todos.loc[mejor, "captación en prueba"] > todos.loc[regla, "captación en prueba"] else regla
print(f"Ganador en validación: {mejor} | se usa: {elegido}")
print(f"Captación en prueba: {todos.loc[elegido, 'captación en prueba']:.2f} "
      f"| regla de una columna: {todos.loc[regla, 'captación en prueba']:.2f} | sorteo: {todos.loc['sorteo', 'captación en prueba']:.2f}")

**Por qué no la exactitud.** Un modelo que dijera «nadie queda abajo» acertaría en tres de cada cuatro estudiantes y no le serviría de nada a la Secretaría. La captación mide lo que importa: cuántos de los que necesitaban el cupo lo recibieron.

## 5. ¿A quién le falla?

Repartimos los cupos con el candidato elegido y miramos, por grupo, qué fracción de los que quedaron abajo recibió cupo.

In [ ]:
cupos = asignar_cupos(puntajes[elegido], CUPO)
grupos = {"zona": datos["cole_area_ubicacion"].str.lower(),
          "sexo": datos["estu_genero"].map({"F": "mujeres", "M": "hombres"}),
          "jornada": X["nocturna_o_sabatina"].map({1.0: "nocturna o sabatina", 0.0: "diurna"})}
auditoria = pd.concat({nombre: auditar(y, cupos, grupo) for nombre, grupo in grupos.items()})
auditoria.round({"n": 0, "% que quedó abajo": 1, "% con cupo": 1, "captación": 2})

Una captación más baja en un grupo significa que el programa deja por fuera, en proporción, a más estudiantes de ese grupo que lo necesitaban. Esa es una decisión política que el modelo no puede tomar solo: la Secretaría puede, por ejemplo, reservar cupos por zona.

### La decisión en tres frases

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

## 6. El post

Tres láminas con lo que dicen los datos de Caldas. Las cifras salen de las celdas de arriba.

In [ ]:
from taller import carrusel as c
c.preparar_fuentes()
PIE = "Fuente: ICFES, Saber 11 2020-2, en datos.gov.co. Cálculos propios.\nCarlos Manuel Orrego Franco"
zona = paneles["Colegio y zona"].query("n >= 50")["tasa"].round(1)      # grupos con muy pocos estudiantes no van al post
c.lamina_barras(zona.to_dict(), {k: (c.ROJO if "rural" in k else c.AZUL) for k in zona.index}, "SABER 11 EN CALDAS",
                f"En los colegios oficiales rurales,\n{zona.max():.0f} % queda en el cuarto\nmás bajo del país",
                "Porcentaje de estudiantes con percentil nacional 25 o menos, 2020-2.", PIE, "2/4", "icfes_2.png")
plt.show()

In [ ]:
madre_tasa = paneles["Educación de la madre"]["tasa"].round(1)
c.lamina_barras(madre_tasa.to_dict(), {k: c.MORADO for k in madre_tasa.index}, "LA FAMILIA PESA",
                f"Si la madre no estudió, {madre_tasa.iloc[0]:.0f} %\nqueda abajo; si tiene posgrado, {madre_tasa.iloc[-1]:.0f} %",
                "Porcentaje en el cuarto más bajo del país\nsegún la educación de la madre.", PIE, "3/4", "icfes_3.png")
plt.show()

In [ ]:
extremos = pd.concat([municipios.nlargest(4, "tasa"), municipios.nsmallest(4, "tasa")]).sort_values("tasa")
c.lamina_ranking(extremos["tasa"], f"{datos['cole_mcpio_ubicacion'].nunique()} MUNICIPIOS",
                 f"En {extremos['tasa'].idxmax()}, {extremos['tasa'].max():.0f} %.\nEn {extremos['tasa'].idxmin()}, {extremos['tasa'].min():.0f} %",
                 "Porcentaje en el cuarto más bajo del país. Los cuatro\nmunicipios más altos y los cuatro más bajos con 80 o más estudiantes.",
                 PIE, "4/4", "icfes_4.png", resaltar=["Manizales"])
plt.show()

## Bitácora

| Decisión | Alternativa | Por qué |
|---|---|---|
| Cuarto más bajo = percentil nacional 25 o menos | Otro corte | El cupo alcanza para uno de cada cuatro |
| Quitar solo los puntajes en cero | Borrar filas con faltantes | Los que dejan preguntas en blanco no se parecen a los demás |
| Llenar faltantes con la mediana del entrenamiento | La mediana de todos | Usar la prueba para llenar sería mirar el examen |
| Tramos por colegio | Por estudiante | Los compañeros de colegio se parecen |
| Captación como métrica | Exactitud | La exactitud premia decir que nadie queda abajo |

## Cambien una línea

Pongan otro departamento en `DEPARTAMENTO` (por ejemplo `"RISARALDA"` o `"CHOCO"`, como lo escribe el ICFES) y corran todo. ¿Las brechas son las mismas?

In [ ]:
procedencia()